# 03 — Medallion exploration

A presentation-friendly walkthrough of all three layers, end-to-end on the same dataset. Pick the env via `AIDP_ENV` at the top of the next cell; everything else (Snowflake creds, FQNs) comes from `config.yaml`.

| Layer | Lives in | What it looks like | Reading code |
|---|---|---|---|
| **Bronze** | Snowflake — `RAPPI_DEV.BRONZE.ORDER_DIMENSIONS` (dev) / `RAPPI_PROD.BRONZE.ORDER_DIMENSIONS` (prod) | Raw, uppercase, mixed nulls | `spark.read.format('snowflake')...` |
| **Silver** | AIDP catalog — `silver_dev` / `silver_prod` (Delta) | Cleaned, lowercase, with derived columns | `spark.table(...)` |
| **Gold**   | AIDP catalog — `gold_dev` / `gold_prod` (external mount to ADW) | Aggregated business mart, one row per (vertical, country, month) | `spark.table(...)` |

The last section picks one gold row and reconstructs it from silver — proves the pipeline is doing what the mart claims.


## Setup

Cluster requirements: same as the other notebooks (`spark-snowflake_2.12-3.1.1.jar` + `snowflake-jdbc-3.19.0.jar` on the classpath, Delta runtime).

Non-sensitive params come from `assets/aidp/config.yaml`. Snowflake `sfUser` / `sfPassword` come from the AIDP credential named under `snowflake.credential` (resolved at execution by the AIDP runtime — no `import aidputils` needed).


In [ ]:
# ══════════════════ Notebook config ══════════════════════════════════
# CONFIG_PATH : workspace path to assets/aidp/config.yaml. Get it from
#               the AIDP UI: right-click config.yaml → 'Copy path'.
# AIDP_ENV    : which block of config.yaml to load ('dev' or 'prod').
# ─────────────────────────────────────────────────────────────────────
import os, yaml
from pyspark.sql import functions as F

CONFIG_PATH = '/Workspace/sales-orders/assets/aidp/config.yaml'
AIDP_ENV    = os.environ.get('AIDP_ENV', 'dev')
print('env: ', AIDP_ENV)
# ═════════════════════════════════════════════════════════════════════

with open(CONFIG_PATH) as f:
    _cfg_all = yaml.safe_load(f)

if AIDP_ENV not in _cfg_all:
    raise KeyError(f'env "{AIDP_ENV}" not in {CONFIG_PATH}; available: {sorted(_cfg_all)}')

cfg = _cfg_all[AIDP_ENV]

# Snowflake: non-sensitive options from config + user/password from AIDP secret.
_snow_cfg     = cfg['snowflake']
_cred_name    = _snow_cfg['credential']
SNOW          = {k: v for k, v in _snow_cfg.items() if k != 'credential'}
SNOW['sfUser']     = aidputils.secrets.get(name=_cred_name, key='user')
SNOW['sfPassword'] = aidputils.secrets.get(name=_cred_name, key='password')
SNOW['application'] = 'aidp-demo-explore'

BRONZE_TABLE = cfg['source']['table']
SILVER_FQN   = '.'.join(p for p in [cfg['silver']['catalog'], cfg['silver']['schema'], cfg['silver']['table']] if p)
GOLD_FQN     = '.'.join(p for p in [cfg['gold']['catalog'],   cfg['gold']['schema'],   cfg['gold']['table']]   if p)

print(f'bronze  : snowflake://{SNOW["sfDatabase"]}.{SNOW["sfSchema"]}.{BRONZE_TABLE}')
print(f'silver  : {SILVER_FQN}')
print(f'gold    : {GOLD_FQN}')


## 1. Bronze — raw orders in Snowflake

Pull a small projected sample to keep the demo fast. We are reading exactly the columns we want to show — projection pushdown means Snowflake only ships those columns over the wire.


In [ ]:
bronze_sample = (
    spark.read.format('snowflake').options(**SNOW)
         .option('query', f"""
             SELECT
               ORDER_ID, USER_ID, COUNTRY_CODE, VERTICAL,
               ORDER_STATUS, TOTAL_AMOUNT, DISCOUNT, IS_PRIME_USER, ORDER_CREATED_AT
             FROM {BRONZE_TABLE}
             LIMIT 10
         """)
         .load()
)

print('schema (raw, uppercase, untyped strings) ──')
bronze_sample.printSchema()
print('\nsample rows ──')
bronze_sample.show(truncate=False)


In [ ]:
# Aggregates pushed entirely into Snowflake — none of these rows transit to Spark.
bronze_kpis = (
    spark.read.format('snowflake').options(**SNOW)
         .option('query', f"""
             SELECT
               COUNT(*)                                                                 AS total_rows,
               COUNT(DISTINCT COUNTRY_CODE)                                             AS countries,
               COUNT(DISTINCT VERTICAL)                                                 AS verticals,
               MIN(ORDER_CREATED_AT)                                                    AS first_order_at,
               MAX(ORDER_CREATED_AT)                                                    AS last_order_at,
               SUM(CASE WHEN ORDER_STATUS = 'delivered' THEN 1 ELSE 0 END)              AS delivered,
               SUM(CASE WHEN ORDER_STATUS = 'cancelled' THEN 1 ELSE 0 END)              AS cancelled
             FROM {BRONZE_TABLE}
         """)
         .load()
)
bronze_kpis.show(truncate=False)


## 2. Silver — cleaned Delta in the AIDP catalog

Same data, but: lowercase / normalized casing, NULL identity rows dropped, enriched with derived columns: `order_date`, `order_hour`, `revenue_net`, `is_delivered`, `is_cancelled`, `bronze_ingested_at`. Plain Delta — no `partitionBy`.


In [ ]:
silver = spark.table(SILVER_FQN)

print('schema (cleaned, lowercase, with derived columns) ──')
silver.printSchema()
print(f'\nrow count: {silver.count():,}')
print('\nsample rows (note: vertical/order_status are lowercase; revenue_net & is_delivered are derived) ──')
silver.select(
    'order_id', 'country_code', 'vertical', 'order_date',
    'order_status', 'is_delivered', 'total_amount', 'discount', 'revenue_net',
).show(10, truncate=False)


In [ ]:
# Top revenue countries.
(silver.filter(F.col('is_delivered'))
       .groupBy('country_code')
       .agg(F.sum('revenue_net').cast('decimal(18,2)').alias('revenue_net_total'),
            F.count('*').alias('delivered_orders'))
       .orderBy(F.col('revenue_net_total').desc())
       .show(truncate=False))


## 3. Gold — vertical performance mart in ADW

Pre-aggregated business mart. Grain: `(vertical, country_code, order_month)`. Materialized in ADW via the AIDP external-catalog mount (`gold_dev` / `gold_prod`) — same Spark catalog API, just a different catalog name in the FQN.


In [ ]:
gold = spark.table(GOLD_FQN)

print('schema ──')
gold.printSchema()
print(f'\nrow count: {gold.count():,}  (≈ verticals × countries × months)')
print('\ntop 15 by GMV ──')
gold.orderBy(F.col('gmv').desc()).show(15, truncate=False)


In [ ]:
# Vertical-level rollup (across all countries & months) — quick health view of the mart.
(gold.groupBy('vertical')
     .agg(F.sum('orders').alias('orders'),
          F.sum('gmv').cast('decimal(18,2)').alias('gmv_total'),
          F.avg('aov').cast('decimal(18,2)').alias('avg_aov'),
          F.avg('cancel_rate').cast('decimal(5,4)').alias('avg_cancel_rate'),
          F.avg('prime_order_share').cast('decimal(5,4)').alias('avg_prime_share'))
     .orderBy(F.col('gmv_total').desc())
     .show(truncate=False))


## 4. End-to-end lineage check

Pick the top-GMV row from gold and recompute the same metrics from silver. If the medallion is doing its job, the numbers match to the cent.


In [ ]:
# 1) pick the top-GMV slice (vertical, country, month) from gold
pick = gold.orderBy(F.col('gmv').desc()).limit(1).collect()[0]
print(f'picked slice: vertical={pick.vertical!r}  country={pick.country_code!r}  month={pick.order_month}')
print(f'gold says     orders={pick.orders}  delivered={pick.delivered_orders}  gmv={pick.gmv}  aov={pick.aov}')

# 2) recompute the same metrics from silver
recomputed = (
    silver
    .filter((F.col('vertical') == pick.vertical) &
            (F.col('country_code') == pick.country_code) &
            (F.trunc('order_date', 'month') == pick.order_month))
    .agg(
        F.count('*').alias('orders'),
        F.sum(F.when(F.col('is_delivered'), 1).otherwise(0)).alias('delivered'),
        F.sum(F.when(F.col('is_delivered'), F.col('total_amount')).otherwise(0))
         .cast('decimal(18,2)').alias('gmv'),
        F.avg(F.when(F.col('is_delivered'), F.col('total_amount')))
         .cast('decimal(18,2)').alias('aov'),
    )
    .collect()[0]
)
print(f'silver recomputes orders={recomputed.orders}  delivered={recomputed.delivered}  gmv={recomputed.gmv}  aov={recomputed.aov}')

ok = (
    recomputed.orders   == pick.orders and
    recomputed.delivered == pick.delivered_orders and
    recomputed.gmv      == pick.gmv
)
print('\n✓ MATCH — gold mart agrees with silver source' if ok else '\n✗ MISMATCH — investigate')
